In [ ]:
# For each OG, cross-reference miniprot GFF Frameshift annotations against the
# seqIDmapping table to get frameshift presence/absence per sequence.
# author: Sheng-Kai Hsu
# (two independent miniprot batches - main + additionalOGs - each with its own
#  MPID numbering; MPID is not comparable across batches, so each is processed
#  through its own consistent GFF-dir/seqIDmapping pairing and written to its
#  own output file, matching how 08A_masterDataTableGeneration.ipynb already
#  reads and rbinds the two)
rm(list=ls())
PHYLOGWAS_ROOT <- Sys.getenv("PHYLOGWAS_ROOT", unset = "/workdir/sh2246/p_phyloGWAS")

library(Biostrings)
library(parallel)
library(rtracklayer)

# main batch

In [ ]:
trslTab = data.table::fread(file.path(PHYLOGWAS_ROOT, "output/seqIDmapping.txt"))
trslTab$mapID = paste(trslTab$MPID,gsub(".fa","",trslTab$inputFile),sep = ":")

gffFlist = list.files(file.path(PHYLOGWAS_ROOT, "output/orthofinderMiniProt"),full.names = T)

FSSeq = unlist(mclapply(gffFlist,function(x){
    gffTest = readGFF(x)
    tmp = gffTest[gffTest$type %in% "mRNA",]
    return(paste(tmp$ID[!is.na(tmp$Frameshift)],gsub('.gff','',basename(x)),sep = ":"))
},mc.cores = 20))

outTab = trslTab[,3:4]
outTab$Frameshift = as.numeric(outTab$mapID%in%FSSeq)

data.table::fwrite(outTab,file.path(PHYLOGWAS_ROOT, "output/frameShiftMutation.txt"),sep = "\t")

# additionalOGs batch

In [ ]:
trslTab = data.table::fread(file.path(PHYLOGWAS_ROOT, "output/seqIDmapping_additionalOGs.txt"))
trslTab$mapID = paste(trslTab$MPID,gsub(".fa","",trslTab$inputFile),sep = ":")

gffFlist = list.files(file.path(PHYLOGWAS_ROOT, "output/orthofinderMiniProt_additionalOGs"),full.names = T)

FSSeq = unlist(mclapply(gffFlist,function(x){
    gffTest = readGFF(x)
    tmp = gffTest[gffTest$type %in% "mRNA",]
    return(paste(tmp$ID[!is.na(tmp$Frameshift)],gsub('.gff','',basename(x)),sep = ":"))
},mc.cores = 20))

outTab = trslTab[,3:4]
outTab$Frameshift = as.numeric(outTab$mapID%in%FSSeq)

data.table::fwrite(outTab,file.path(PHYLOGWAS_ROOT, "output/frameShiftMutation_additionalOGs.txt"),sep = "\t")